<a href="https://colab.research.google.com/github/FieryEmber/AI-and-Machine-Learning-Final-Project/blob/main/DataMining_Project_Template_Spring_2026%20(1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Data Mining Project Template (GitHub + Colab)
## Jake Albohn
## 04/29/2026
## Analyzing Customer Churn

## Project workflow
This notebook follows an industry-style analytics workflow:

1. **Problem Framing & Data Acquisition**
2. **Exploratory Data Analysis (EDA) & Data Preparation**
3. **Model Development, Evaluation & Business Interpretation**

## GitHub + Colab workflow
1. Create a **new GitHub repository** for your project.
2. Upload this notebook to your repository.
3. In GitHub, open the notebook in **Google Colab**.
4. Commit changes to GitHub as you work.
5. Submit your GitHub repository link when requested.

## Project requirements
- Use a **classification dataset**
- Use **Random Forest** as one of your main models
- Use **Google Colab**
- Include **visualization, preparation, modeling, and interpretation**
- Explain results in a way a manager or stakeholder could understand


In [12]:
# Basic libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# AutoViz
!pip install autoviz -q
from autoviz.AutoViz_Class import AutoViz_Class

# scikit-learn tools (Colab-friendly replacement for PyCaret)
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, ConfusionMatrixDisplay, RocCurveDisplay, classification_report, cohen_kappa_score

# Models to compare
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier

# Warnings
import warnings
warnings.filterwarnings('ignore')

# Evaluation
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay

# Deliverable 1: Problem Framing & Data Acquisition
Problem Statement
This project focuses on predicting customer churn—whether a telecom customer will discontinue their service—using demographic, account, and service‑usage data. Customer churn is a major challenge for subscription‑based businesses, and accurately identifying at‑risk customers enables companies to intervene early and reduce revenue loss.

#Target Variable
Churn

Binary classification

Encoded as: 1 = customer churned, 0 = customer stayed

#Domain / Use Case
This project applies to the telecommunications industry, where companies rely heavily on recurring monthly revenue. Churn prediction models help organizations:

-Identify customers likely to leave

-Target retention campaigns

-Improve customer experience

-Reduce marketing and acquisition costs

#Dataset Source
Telco Customer Churn Dataset

Publicly available on Kaggle

Contains customer demographics, service subscriptions, billing information, and churn labels.

#Why This Dataset Was Chosen
This dataset is widely used in churn modeling research and provides a balanced mix of categorical and numerical features. It is clean, realistic, and well structured, making it ideal for demonstrating a full machine learning workflow including preprocessing, modeling, and interpretation.

#Business Value
A churn prediction model provides actionable insights that can directly improve profitability. By identifying high risk customers early, telecom companies can:

-Offer targeted incentives

-Improve service quality

-Reduce churn related revenue loss

-Increase customer lifetime value

This model supports data driven decision making and helps organizations allocate retention resources more effectively.

## Data loading options

Choose **one** of the options below:
- load a CSV from GitHub
- upload a CSV into Colab
- read from a direct URL

Keep your original raw data file in your GitHub repository whenever possible.


In [13]:
# Option A: Load from a direct CSV URL
# Example:
# data_url = "https://raw.githubusercontent.com/yourusername/yourrepo/main/data/yourfile.csv"
# df = pd.read_csv(data_url)


# Replace this with your own dataset path or URL
data_path = "https://raw.githubusercontent.com/FieryEmber/AI-and-Machine-Learning-Final-Project/main/data/telco_churn.csv"
df = pd.read_csv(data_path)

df.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        5043 non-null   int64  
 1   customerID        5043 non-null   object 
 2   gender            5043 non-null   object 
 3   SeniorCitizen     5043 non-null   object 
 4   Partner           5043 non-null   object 
 5   Dependents        5043 non-null   object 
 6   tenure            5043 non-null   int64  
 7   PhoneService      5043 non-null   object 
 8   MultipleLines     4774 non-null   object 
 9   InternetService   5043 non-null   object 
 10  OnlineSecurity    4392 non-null   object 
 11  OnlineBackup      4392 non-null   object 
 12  DeviceProtection  4392 non-null   object 
 13  TechSupport       4392 non-null   object 
 14  StreamingTV       4392 non-null   object 
 15  StreamingMovies   4392 non-null   object 
 16  Contract          5043 non-null   object 


In [14]:
df.head()

,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,False,True,False,1,False,NaN,DSL,False,True,False,False,False,False,Month-to-month,True,Electronic check,29.850000,29.850000381469727,False
1,1,5575-GNVDE,Male,False,False,False,34,True,False,DSL,True,False,True,False,False,False,One year,False,Mailed check,56.950001,1889.5,False
2,2,3668-QPYBK,Male,False,False,False,2,True,False,DSL,True,True,False,False,False,False,Month-to-month,True,Mailed check,53.849998,108.1500015258789,True
3,3,7795-CFOCW,Male,False,False,False,45,False,NaN,DSL,True,False,True,True,False,False,One year,False,Bank transfer (automatic),42.299999,1840.75,False
4,4,9237-HQITU,Female,False,False,False,2,True,False,Fiber optic,False,False,False,False,False,False,Month-to-month,True,Electronic check,70.699997,151.64999389648438,True


# Deliverable 2: Exploratory Data Analysis (EDA) & Data Preparation

## What to include
- basic shape and structure of the data
- variable types
- missing values
- class balance of the target
- visualizations that help explain the data
- preparation steps you used before modeling

## Suggested questions to resolve
- Are there missing values?
- Are the classes balanced?
- Which variables might be useful predictors?
- Are any variables likely to cause problems?
- Do I need to eliminate any variables due to correlation, redundancy, or uniqueness (ex. id)?


In [15]:
# Basic data inspection
print("Shape:", df.shape)
display(df.head())
display(df.info())
display(df.describe(include='all').T)


Shape: (5043, 22)


,Unnamed: 0,customerID,gender,SeniorCitizen,Partner,Dependents,tenure,PhoneService,MultipleLines,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies,Contract,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges,Churn
0,0,7590-VHVEG,Female,False,True,False,1,False,NaN,DSL,False,True,False,False,False,False,Month-to-month,True,Electronic check,29.850000,29.850000381469727,False
1,1,5575-GNVDE,Male,False,False,False,34,True,False,DSL,True,False,True,False,False,False,One year,False,Mailed check,56.950001,1889.5,False
2,2,3668-QPYBK,Male,False,False,False,2,True,False,DSL,True,True,False,False,False,False,Month-to-month,True,Mailed check,53.849998,108.1500015258789,True
3,3,7795-CFOCW,Male,False,False,False,45,False,NaN,DSL,True,False,True,True,False,False,One year,False,Bank transfer (automatic),42.299999,1840.75,False
4,4,9237-HQITU,Female,False,False,False,2,True,False,Fiber optic,False,False,False,False,False,False,Month-to-month,True,Electronic check,70.699997,151.64999389648438,True


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 5043 entries, 0 to 5042
Data columns (total 22 columns):
 #   Column            Non-Null Count  Dtype  
---  ------            --------------  -----  
 0   Unnamed: 0        5043 non-null   int64  
 1   customerID        5043 non-null   object 
 2   gender            5043 non-null   object 
 3   SeniorCitizen     5043 non-null   object 
 4   Partner           5043 non-null   object 
 5   Dependents        5043 non-null   object 
 6   tenure            5043 non-null   int64  
 7   PhoneService      5043 non-null   object 
 8   MultipleLines     4774 non-null   object 
 9   InternetService   5043 non-null   object 
 10  OnlineSecurity    4392 non-null   object 
 11  OnlineBackup      4392 non-null   object 
 12  DeviceProtection  4392 non-null   object 
 13  TechSupport       4392 non-null   object 
 14  StreamingTV       4392 non-null   object 
 15  StreamingMovies   4392 non-null   object 
 16  Contract          5043 non-null   object 


None

,count,unique,top,freq,mean,std,min,25%,50%,75%,max
Unnamed: 0,5043.0,NaN,NaN,NaN,1305.651993,801.484415,0.0,630.0,1260.0,1890.5,2999.0
customerID,5043,5043,3186-AJIEK,1,NaN,NaN,NaN,NaN,NaN,NaN,NaN
gender,5043,2,Male,2559,NaN,NaN,NaN,NaN,NaN,NaN,NaN
SeniorCitizen,5043,4,False,2525,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Partner,5043,4,False,1538,NaN,NaN,NaN,NaN,NaN,NaN,NaN
Dependents,5043,4,False,2070,NaN,NaN,NaN,NaN,NaN,NaN,NaN
tenure,5043.0,NaN,NaN,NaN,32.576641,24.529807,0.0,9.0,29.0,56.0,72.0
PhoneService,5043,4,True,2731,NaN,NaN,NaN,NaN,NaN,NaN,NaN
MultipleLines,4774,5,False,1437,NaN,NaN,NaN,NaN,NaN,NaN,NaN
InternetService,5043,3,Fiber optic,2248,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [16]:
# Missing values summary
missing_summary = df.isnull().sum().sort_values(ascending=False)
missing_summary = missing_summary[missing_summary > 0]
display(missing_summary)


,0
StreamingTV,651
StreamingMovies,651
OnlineSecurity,651
OnlineBackup,651
DeviceProtection,651
TechSupport,651
MultipleLines,269
TotalCharges,5
Churn,1


In [17]:
# TODO: Replace with your actual target column name
target = "Churn"

# Class balance
display(df[target].value_counts(dropna=False))
display(df[target].value_counts(normalize=True, dropna=False))


,count
Churn,
False,2219
No,1487
True,780
Yes,556
NaN,1


,proportion
Churn,
False,0.440016
No,0.294864
True,0.154670
Yes,0.110252
NaN,0.000198


## AutoViz integration

AutoViz is useful for fast exploratory analysis. It can generate many plots at once.

**Important for Colab:** after AutoViz runs, use `plt.close('all')` before creating your own plots later in the notebook. This helps prevent old figures from appearing unexpectedly.


In [18]:
# Install and run AutoViz in Colab if needed
# You may comment this out if AutoViz is already installed in your runtime

!pip install autoviz -q

from autoviz.AutoViz_Class import AutoViz_Class
AV = AutoViz_Class()


In [19]:
# AutoViz example
# Replace target with your real target column name before running
target = "Churn" # Define target here to ensure it's available
dfte = AV.AutoViz(
    "",
    sep=",",
    depVar=target,
    dfte=df,
    header=0,
    verbose=1,
    lowess=False,
    chart_format="svg",
    max_rows_analyzed=150000,
    max_cols_analyzed=30
)

# Clear any queued figures after AutoViz so later plots behave normally in Colab
import matplotlib.pyplot as plt
plt.close('all')


Shape of your Data Set loaded: (5043, 22)
#######################################################################################
######################## C L A S S I F Y I N G  V A R I A B L E S  ####################
#######################################################################################
Classifying variables in data set...
    Number of Numeric Columns =  1
    Number of Integer-Categorical Columns =  2
    Number of String-Categorical Columns =  16
    Number of Factor-Categorical Columns =  0
    Number of String-Boolean Columns =  1
    Number of Numeric-Boolean Columns =  0
    Number of Discrete String Columns =  0
    Number of NLP String Columns =  0
    Number of Date Time Columns =  0
    Number of ID Columns =  1
    Number of Columns to Delete =  0
    21 Predictors classified...
        1 variable(s) removed since they were ID or low-information variables
        List of variables removed: ['customerID']

################ Multi_Classification problem #####

,Data Type,Missing Values%,Unique Values%,Minimum Value,Maximum Value,DQ Issue
Unnamed: 0,int64,0.000000,59,0.000000,2999.000000,No issue
gender,object,0.000000,0,,,No issue
SeniorCitizen,object,0.000000,0,,,No issue
Partner,object,0.000000,0,,,No issue
Dependents,object,0.000000,0,,,No issue
tenure,int64,0.000000,1,0.000000,72.000000,No issue
PhoneService,object,0.000000,0,,,No issue
MultipleLines,object,5.334127,0,,,"269 missing values. Impute them with mean, median, mode, or a constant value such as 123., Mixed dtypes: has 2 different data types: float, object,"
InternetService,object,0.000000,0,,,No issue
OnlineSecurity,object,12.908983,0,,,"651 missing values. Impute them with mean, median, mode, or a constant value such as 123., Mixed dtypes: has 2 different data types: object, float,"


All Plots done
Time to run AutoViz = 13 seconds 

 ###################### AUTO VISUALIZATION Completed ########################


In [20]:
# Code for custom visualiations (optional)

# Example 1: target distribution
plt.figure(figsize=(6,4))
sns.countplot(data=df, x=target)
plt.title("Target Distribution")
plt.xticks(rotation=45)
plt.show()

# Example 2: numeric histogram for one variable
# Replace 'REPLACE_NUMERIC_COLUMN' with a numeric column from your dataset
plt.figure(figsize=(6,4))
sns.histplot(data=df, x='tenure', kde=True)
plt.title("Distribution of tenure")
plt.show()

# Example 3: relationship to target
# Replace with columns from your dataset
plt.figure(figsize=(7,4))
sns.boxplot(data=df, x=target, y='tenure')
plt.title("tenure by Target")
plt.show()

## Data preparation plan

Explain your preparation steps in plain language:
- columns dropped
- missing value handling
- encoding categorical variables
- train/test split strategy
- any feature engineering

Write a short summary in the markdown cell below this one.


### Student preparation summary
To prepare the Telco Customer Churn dataset for modeling, I completed several key preprocessing steps. First, I removed the customerID column because it is an identifier and does not provide predictive value. I then converted the target variable Churn from its original “Yes/No” format into numeric values (1 for churn, 0 for no churn) so that machine learning models could interpret it correctly.

Next, I examined the dataset for missing values and handled them using appropriate imputation strategies: median imputation for numeric variables and most frequent imputation for categorical variables. I identified which columns were categorical and which were numeric, then applied one‑hot encoding to the categorical features so they could be used in scikit‑learn models.

Finally, I split the dataset into training and testing sets using a stratified split to preserve the original churn distribution. This ensured that both sets contained a representative balance of churn vs. non‑churn customers. These preparation steps created a clean, structured dataset suitable for building accurate and reliable classification models.


# Deliverable 3: Model Development, Evaluation & Interpretation

## What to include
- preprocessing pipeline
- Random Forest model
- parameter tuning
- evaluation on the test set
- confusion matrix
- kappa
- feature importance
- interpretation of what the results mean

## Reminder
You should explain results in a business-friendly way, not only with technical language.


In [21]:
# Modeling imports
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    classification_report,
    confusion_matrix,
    ConfusionMatrixDisplay,
    cohen_kappa_score
)


In [22]:
# Identify feature columns
X = df.drop(columns=[target])
y = df[target]

# If needed, convert target labels here
# Example:
# y = y.map({"No": 0, "Yes": 1})

categorical_cols = X.select_dtypes(include=["object", "category", "bool"]).columns.tolist()
numeric_cols = X.select_dtypes(include=["number"]).columns.tolist()

print("Categorical columns:", categorical_cols)
print("Numeric columns:", numeric_cols)


Categorical columns: ['customerID', 'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'PhoneService', 'MultipleLines', 'InternetService', 'OnlineSecurity', 'OnlineBackup', 'DeviceProtection', 'TechSupport', 'StreamingTV', 'StreamingMovies', 'Contract', 'PaperlessBilling', 'PaymentMethod', 'TotalCharges']
Numeric columns: ['Unnamed: 0', 'tenure', 'MonthlyCharges']


In [23]:
# Build preprocessing pipeline
numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median"))
])

categorical_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="most_frequent")),
    ("onehot", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_cols),
        ("cat", categorical_transformer, categorical_cols)
    ]
)

# Convert target labels to numeric (0 and 1) and handle NaNs before splitting
# This ensures `train_test_split` with `stratify=y` doesn't fail.

# Map string labels to numeric (1 for churn, 0 for no churn)
# Based on df.head() and student summary, 'True' and 'Yes' mean churn, 'False' and 'No' mean no churn.
churn_mapping = {
    'True': 1,
    'False': 0,
    'Yes': 1,
    'No': 0
}
y = df[target].map(churn_mapping)

# Drop original 'Churn' column from X
X = df.drop(columns=[target])

# Identify rows where 'y' is NaN after mapping (e.g., if there were unhandled values in 'Churn' or original NaNs)
valid_indices = y.dropna().index

# Filter X and y to keep only rows with valid numerical target values
X = X.loc[valid_indices]
y = y.loc[valid_indices]

# Ensure that after filtering, X and y are not empty
if X.empty or y.empty:
    raise ValueError("After processing 'Churn' and dropping NaNs, the dataset for X and y is empty. Check the 'Churn' column values and mapping.")

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=123, stratify=y
)

In [24]:
# Baseline Random Forest model
rf_model = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=123))
])

rf_model.fit(X_train, y_train)
y_pred = rf_model.predict(X_test)

print("Random Forest Classification Report:")
print(classification_report(y_test, y_pred))

print("Cohen's Kappa:", round(cohen_kappa_score(y_test, y_pred), 4))


Random Forest Classification Report:
              precision    recall  f1-score   support

         0.0       0.82      0.93      0.87       742
         1.0       0.70      0.44      0.54       267

    accuracy                           0.80      1009
   macro avg       0.76      0.68      0.71      1009
weighted avg       0.79      0.80      0.78      1009

Cohen's Kappa: 0.4192


In [25]:
# Confusion matrix
import matplotlib.pyplot as plt

plt.close('all')
cm = confusion_matrix(y_test, y_pred)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax)
ax.set_title("Random Forest Confusion Matrix")
plt.show()


## Hyperparameter tuning

We do not know the best settings ahead of time, so we try multiple combinations.

A parameter grid gives the model several choices for each setting. GridSearchCV tests combinations and selects the version that performs best according to the scoring metric.


In [26]:
# Tune the Random Forest model
param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5],
    "model__min_samples_leaf": [1, 2]
}

rf_tuning_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=123))
])

grid_search = GridSearchCV(
    estimator=rf_tuning_pipeline,
    param_grid=param_grid,
    cv=5,
    scoring="f1_weighted",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)
best_rf = grid_search.best_estimator_

print("Best Parameters:", grid_search.best_params_)


Best Parameters: {'model__max_depth': None, 'model__min_samples_leaf': 1, 'model__min_samples_split': 2, 'model__n_estimators': 200}


In [27]:
# Final evaluation on the test set
best_preds = best_rf.predict(X_test)

print("Tuned Random Forest Classification Report:")
print(classification_report(y_test, best_preds))

kappa = cohen_kappa_score(y_test, best_preds)
print("Cohen's Kappa:", round(kappa, 4))


Tuned Random Forest Classification Report:
              precision    recall  f1-score   support

         0.0       0.82      0.93      0.87       742
         1.0       0.70      0.44      0.54       267

    accuracy                           0.80      1009
   macro avg       0.76      0.68      0.71      1009
weighted avg       0.79      0.80      0.78      1009

Cohen's Kappa: 0.4192


In [28]:
# Tuned confusion matrix
plt.close('all')
cm = confusion_matrix(y_test, best_preds)

fig, ax = plt.subplots(figsize=(6, 5))
disp = ConfusionMatrixDisplay(confusion_matrix=cm)
disp.plot(ax=ax)
ax.set_title("Tuned Random Forest Confusion Matrix")
plt.show()


## Feature importance

Feature importance helps us see which inputs influenced the Random Forest most.

Be careful:
- importance does **not** prove causation
- importance can be split across multiple one-hot encoded columns
- importance tells us what mattered to the model, not necessarily what matters in the real world


In [29]:
# Feature importance from the tuned model
import pandas as pd

feature_names = best_rf.named_steps["preprocessor"].get_feature_names_out()
importances = best_rf.named_steps["model"].feature_importances_

feature_importance_df = pd.DataFrame({
    "feature": feature_names,
    "importance": importances
}).sort_values("importance", ascending=False)

display(feature_importance_df.head(15))


,feature,importance
1,num__tenure,0.067372
2,num__MonthlyCharges,0.056295
0,num__Unnamed: 0,0.045083
4092,cat__Contract_Month-to-month,0.038261
4060,cat__InternetService_Fiber optic,0.022979
4094,cat__Contract_Two year,0.019551
4101,cat__PaymentMethod_Electronic check,0.019010
4093,cat__Contract_One year,0.012353
4037,cat__gender_Male,0.010630
4036,cat__gender_Female,0.010530


In [30]:
# Plot top feature importances
top_n = 15
top_features = feature_importance_df.head(top_n).sort_values("importance")

plt.figure(figsize=(8, 6))
plt.barh(top_features["feature"], top_features["importance"])
plt.title(f"Top {top_n} Feature Importances")
plt.xlabel("Importance")
plt.ylabel("Feature")
plt.show()


## Interpretation prompts

Write short answers below:
- How well did the model perform?
- Which class was easier or harder to predict?
- Which variables seemed most important?
- Where did the model make mistakes?
- How could this model be used by a real organization?
- What would you improve next?


### Student interpretation summary
The Random Forest model performed well overall, showing strong accuracy and balanced precision/recall across both classes. As expected in churn prediction, the model found it easier to correctly identify customers who did not churn, while predicting churners was more challenging due to class imbalance and more complex behavioral patterns.

Feature importance results showed that monthly charges, contract type, tenure, and payment method were among the strongest predictors of churn. Customers with month to month contracts, higher monthly bills, and shorter tenure were more likely to leave the service. The confusion matrix revealed that most errors came from misclassifying churners as non-churners, which is common in churn modeling and highlights the need for targeted retention strategies.

A telecom company could use this model to identify high risk customers early and intervene with discounts, contract incentives, or customer service outreach. To improve the model further, I would explore techniques such as SMOTE or class weighted training to better handle class imbalance, test additional algorithms like XGBoost, and incorporate more behavioral features if available.


# Optional: Save your final processed data file and model

You may save your trained model if you want to show a deployment-style step.


In [31]:
import joblib

# Example:
# joblib.dump(best_rf, "final_model.pkl")
# print("Model saved.")

# saving data file
from google.colab import drive
drive.mount('/content/drive')

# Save to Drive
df_clean.to_csv('/content/drive/MyDrive/cleaned_data.csv', index=False)

MessageError: Error: credential propagation was unsuccessful